In [ ]:
# -*- coding: utf-8 -*-
"""
CSI1800 (HS300 + CSI500 + CSI1000) monthly-rebalance multifactor stock selection
- Universe: CSI1800 constituents (union of 000300.SH, 000905.SH, 000852.SH)
- Factors (top-level): Value, LowPrice, LowAttention, Improvement
- Processing: winsorize -> zscore (industry neutral for most) -> subfactor equal-weight
- Top weights: rolling IC/IR weighting (Spearman IC), default 12 months window
- Rebalance: monthly (last trading day of each month)
- Backtest: monthly returns (equal weight in top N stocks), turnover + optional cost

Author: ZhangYL
"""

'\nCSI1800 (HS300 + CSI500 + CSI1000) monthly-rebalance multifactor stock selection\n- Universe: CSI1800 constituents (union of 000300.SH, 000905.SH, 000852.SH)\n- Factors (top-level): Value, LowPrice, LowAttention, Improvement\n- Processing: winsorize -> zscore (industry neutral for most) -> subfactor equal-weight\n- Top weights: rolling IC/IR weighting (Spearman IC), default 12 months window\n- Rebalance: monthly (last trading day of each month)\n- Backtest: monthly returns (equal weight in top N stocks), turnover + optional cost\n\nAuthor: ZhangYulin\n'

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import statsmodels.api as sm
from scipy.stats import spearmanr

from WindPy import w

ModuleNotFoundError: No module named 'WindPy'

## Wind helpers

In [ ]:
def wind_start():
    if not w.isconnected():
        w.start()
    if not w.isconnected():
        raise RuntimeError("WindPy not connected. Please login Wind terminal first.")


def chunk_list(x: List[str], n: int = 500) -> List[List[str]]:
    return [x[i:i+n] for i in range(0, len(x), n)]


def to_datestr(dt) -> str:
    if hasattr(dt, "strftime"):
        return dt.strftime("%Y-%m-%d")
    return str(dt)[:10]


def wind_tdays(start: str, end: str) -> List[str]:
    d = w.tdays(start, end, "")
    if d.ErrorCode != 0 or not d.Data:
        raise RuntimeError(f"tdays error={d.ErrorCode}")
    return [to_datestr(x) for x in d.Data[0]]


def wind_tdaysoffset(offset: int, date: str) -> str:
    d = w.tdaysoffset(offset, date, "")
    if d.ErrorCode != 0 or not d.Data:
        raise RuntimeError(f"tdaysoffset error={d.ErrorCode}")
    return to_datestr(d.Data[0][0])


def month_end_trading_days(start: str, end: str) -> List[str]:
    days = pd.to_datetime(wind_tdays(start, end))
    df = pd.DataFrame({"d": days})
    df["m"] = df["d"].dt.to_period("M")
    me = df.groupby("m")["d"].max().sort_values()
    return [x.strftime("%Y-%m-%d") for x in me.values]


def wset_index_constituents(index_code: str, date: str) -> List[str]:
    r = w.wset("sectorconstituent", f"date={date};windcode={index_code};field=wind_code")
    if r.ErrorCode != 0:
        print(f"Warning: wset error code {r.ErrorCode} for {index_code} at {date}")
        return []
    if not r.Data:
        return []
    return list(pd.Series(r.Data[0]).dropna().astype(str).unique())


def get_universe_csi1800(date: str, index_codes: Tuple[str, ...]) -> List[str]:
    u = set()
    for idx in index_codes:
        codes = wset_index_constituents(idx, date)
        if codes:
            u |= set(codes)
    return sorted(list(u))


def wss_df(codes: List[str], fields: List[str], options: str = "") -> pd.DataFrame:
    """
    Robust w.wss wrapper -> DataFrame (index=codes, columns=fields)
    """
    if len(codes) == 0:
        return pd.DataFrame(index=[], columns=fields)

    frames = []
    for part in chunk_list(codes, 800):
        try:
            ec, df = w.wss(",".join(part), ",".join(fields), options, usedf=True)
            if ec != 0:
                print(f"Warning: wss error={ec} fields={fields}")
                continue
            
            # df columns are uppercase by WindPy usedf
            df = df.copy()
            df.index = df.index.astype(str)
            df.columns = [c.upper() for c in df.columns]
            frames.append(df)
        except Exception as e:
            print(f"Error in wss_df chunk: {e}")
            continue

    if not frames:
        return pd.DataFrame(index=codes, columns=fields)

    out = pd.concat(frames, axis=0)
    # Ensure requested order and columns (handle missing columns if API partially failed)
    col_map = {f.upper(): f for f in fields}
    # Create empty cols if missing
    for f in fields:
        if f.upper() not in out.columns:
            out[f.upper()] = np.nan
            
    out = out[[f.upper() for f in fields]].rename(columns=col_map)
    
    # Reindex to ensure all input codes are in index (fill NaN for missing)
    out = out.reindex(codes)
    return out


def wsd_df(codes: List[str], field: str, start: str, end: str, options: str = "") -> pd.DataFrame:
    """
    w.wsd wrapper returning DataFrame:
    index = dates, columns = codes
    """
    if len(codes) == 0:
        return pd.DataFrame()

    # Wind wsd 单字段多代码最稳
    ec, df = w.wsd(",".join(codes), field, start, end, options, usedf=True)
    if ec != 0:
        print(f"Warning: wsd error={ec} field={field}")
        return pd.DataFrame()
    # df: index=dates, columns=[FIELD]?? or columns=codes depending on WindPy version
    # normalize:
    if isinstance(df.columns, pd.MultiIndex):
        # not expected in most setups
        df.columns = [c[0] for c in df.columns]
    # Some WindPy returns columns like ['CLOSE'] only; but index might be code-level.
    # We'll handle common case: df has one column (FIELD) but codes embedded in df columns?:
    if len(codes) == 1 and df.shape[1] == 1:
        df.columns = codes
    return df

def pick_first_valid_field(codes: List[str], candidates: List[str], options: str) -> Optional[str]:
    if not codes:
        return None
        
    test_codes = codes[:5]
    for f in candidates:
        try:
            df = wss_df(test_codes, [f], options)
            s = df[f]
            if s.notna().sum() >= 1:
                return f
        except Exception:
            continue
    return None

## Data Cleaning

In [ ]:
def winsorize_series(s: pd.Series, q: float = 0.01) -> pd.Series:
    if s.dropna().empty:
        return s
    lo, hi = s.quantile(q), s.quantile(1 - q)
    return s.clip(lower=lo, upper=hi)


def zscore(s: pd.Series) -> pd.Series:
    mu = s.mean(skipna=True)
    sd = s.std(skipna=True, ddof=0)
    if sd == 0 or np.isnan(sd):
        return s * 0.0
    return (s - mu) / sd


def group_winsorize_z(df: pd.DataFrame, col: str, group_col: str, q: float) -> pd.Series:
    def _proc(g):
        x = winsorize_series(g[col], q=q)
        return zscore(x)
    return df.groupby(group_col, group_keys=False).apply(_proc)

# Neutralization regression

In [ ]:
def neutralize_residual(
    df: pd.DataFrame,
    y_col: str,
    size_col: str,
    industry_col: str,
    min_ind_n: int = 10
) -> pd.Series:
    """
    Cross-sectional OLS each month:
    y ~ ln(mktcap) + industry dummies
    return residuals
    """
    x = df[[y_col, size_col, industry_col]].copy()
    x = x.dropna()

    if x.empty:
        return pd.Series(index=df.index, dtype=float)

    # merge small industries
    ind_counts = x[industry_col].value_counts()
    small_inds = set(ind_counts[ind_counts < min_ind_n].index)
    x[industry_col] = x[industry_col].apply(lambda v: "OTHER" if v in small_inds else v)

    # design matrix
    y = x[y_col].astype(float)
    X = pd.DataFrame({
        "ln_size": np.log(x[size_col].astype(float).clip(lower=1.0))
    }, index=x.index)
    dummies = pd.get_dummies(x[industry_col], prefix="ind", drop_first=True)
    X = pd.concat([X, dummies], axis=1)
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()
    resid = y - model.predict(X)

    out = pd.Series(index=df.index, dtype=float)
    out.loc[resid.index] = resid
    return out

# Factor construction

In [ ]:
def classify_financial_by_sw1(ind_name: pd.Series) -> pd.Series:
    """
    Very practical rule: SW1 industry names containing '银行' or '非银金融' or '证券' or '保险' -> financial
    """
    s = ind_name.fillna("").astype(str)
    is_fin = s.str.contains("银行") | s.str.contains("非银") | s.str.contains("证券") | s.str.contains("保险") | s.str.contains("金融")
    return is_fin


def safe_inverse(x: pd.Series) -> pd.Series:
    return 1.0 / x.replace(0, np.nan)

def calc_ts_rank(current_val: pd.Series, history_df: pd.DataFrame) -> pd.Series:
    """
    计算当前值在历史序列中的百分位
    current_val: index=code
    history_df: index=date, columns=code (包含过去N个月的数据)
    
    Return:
    Series (index=code), 1 means current is highest in history, 0 means lowest.
    """
    # 将当前值拼接到历史数据的最后一行，统一计算Rank
    combined = pd.concat([history_df, current_val.to_frame().T], axis=0)
    
    # pct=True 返回百分位数, method='max' 处理相同数值
    ranks = combined.rank(axis=0, pct=True, method='max')
    
    # 取最后一行的Rank，即当前值的Rank
    current_rank = ranks.iloc[-1]
    return current_rank

def compute_value_factor_optimized(df0: pd.DataFrame, hist_pe: pd.DataFrame, hist_pb: pd.DataFrame, cfg: Config) -> pd.Series:
    """
    低估值=0.7 * (1 - 过去5年PE/PB分位数) + 0.3 * (截面行业内低估值得分)
    """
    df = df0.copy()

    # --- Part A: 短周期/截面 (原逻辑) ---
    df["EP"] = safe_inverse(df["PE_TTM"])
    df["BP"] = safe_inverse(df["PB_LF"])
    df["DIVY"] = df["DIVIDENDYIELD2"]
    df["EBIT_EV"] = df["EBIT_TTM"] / df["EV"].replace(0, np.nan)
    df["CFO_YIELD"] = df["NCF_OPER_TTM"] / df["MKT_CAP"].replace(0, np.nan)

    # 截面 Z-Score
    for c in ["EP", "BP", "DIVY", "EBIT_EV", "CFO_YIELD"]:
        df[c] = df[c].astype(float)
        df[c] = group_winsorize_z(df, c, "IND", q=cfg.winsor_q)

    is_fin = df["IS_FIN"].astype(bool)
    fin_score = df.loc[is_fin, ["BP", "EP", "DIVY"]].mean(axis=1)
    nfin_score = df.loc[~is_fin, ["EP", "EBIT_EV", "CFO_YIELD"]].mean(axis=1)

    score_cs = pd.Series(index=df.index, dtype=float)
    score_cs.loc[is_fin] = fin_score
    score_cs.loc[~is_fin] = nfin_score
    score_cs = zscore(winsorize_series(score_cs, q=cfg.winsor_q))

    # --- Part B: 长周期/纵向 (新逻辑) ---
    # 计算当前PE和PB在各自过去N个月中的分位数
    # 注意：Wind PE/PB 可能有负值，直接做 Rank 即可，负值会被排在最前（视为极度低估/亏损）
    rank_pe = calc_ts_rank(df["PE_TTM"], hist_pe)
    rank_pb = calc_ts_rank(df["PB_LF"], hist_pb)
    
    # 分位数越低，代表越便宜。得分 = 1 - Rank
    avg_rank = (rank_pe + rank_pb) / 2.0
    score_ts = 1.0 - avg_rank
    score_ts = zscore(winsorize_series(score_ts, q=cfg.winsor_q))

    # --- Part C: 组合 ---
    final_score = cfg.weight_long_term * score_ts + cfg.weight_short_term * score_cs
    return final_score


def compute_lowprice_factor_optimized(df0: pd.DataFrame, hist_price: pd.DataFrame, cfg: Config) -> pd.Series:
    """
    低股价：
    0.7 * (1 - 过去5年复权价格分位数) + 0.3 * (截面市值中性化低价)
    """
    df = df0.copy()

    # --- Part A: 短周期/截面 (原逻辑) ---
    # 相对低价：剔除市值和行业影响后的低价
    df["LN_PRICE"] = np.log(df["PRICE"].astype(float).clip(lower=0.01))
    df["RESID"] = neutralize_residual(df, "LN_PRICE", "MKT_CAP", "IND", min_ind_n=cfg.min_industry_n)
    df["RESID"] = winsorize_series(df["RESID"], q=cfg.winsor_q)
    score_cs = -zscore(df["RESID"]) # 残差越小越好

    # --- Part B: 长周期/纵向 (新逻辑) ---
    # 当前价格在过去5年复权价格中的位置
    # 注意：df['PRICE'] 是不复权的，但 hist_price 必须是复权的
    rank_price = calc_ts_rank(df["PRICE"], hist_price)
    
    score_ts = 1.0 - rank_price # 越低越好
    score_ts = zscore(winsorize_series(score_ts, q=cfg.winsor_q))

    # --- Part C: 组合 ---
    final_score = cfg.weight_long_term * score_ts + cfg.weight_short_term * score_cs
    return final_score


def compute_attention_factor(
    df0: pd.DataFrame,
    q: float,
    min_ind_n: int,
    coverage_col: Optional[str],
    report_col: Optional[str]
) -> pd.Series:
    """
    Low attention:
    - Use analyst coverage and report count if available
    - Each: log1p(metric) neutralize on ln(mktcap)+industry dummies -> residual -> z -> negative
    - If both missing, fallback proxy: turnover (TURN) and amount(AMT) as attention proxies.
    """
    df = df0.copy()
    scores = []

    def _one(metric_col: str):
        tmp = df.copy()
        tmp["Y"] = np.log1p(tmp[metric_col].astype(float).clip(lower=0))
        tmp["RES"] = neutralize_residual(tmp, "Y", "MKT_CAP", "IND", min_ind_n=min_ind_n)
        tmp["RES"] = winsorize_series(tmp["RES"], q=q)
        return -zscore(tmp["RES"])

    used = False
    if coverage_col is not None and coverage_col in df.columns:
        scores.append(_one(coverage_col))
        used = True
    if report_col is not None and report_col in df.columns:
        scores.append(_one(report_col))
        used = True

    if not used:
        # fallback proxies (still "attention"): turnover and amount
        # TURN usually in %, AMT in RMB
        if "TURN" in df.columns:
            scores.append(_one("TURN"))
        if "AMT_20D" in df.columns:
            scores.append(_one("AMT_20D"))

    if len(scores) == 0:
        return pd.Series(index=df.index, data=np.nan)

    return pd.concat(scores, axis=1).mean(axis=1)


def compute_improve_factor(df0: pd.DataFrame, q: float) -> pd.Series:
    """
    Improvement:
    ImproveScore = 0.6*ImproveA + 0.4*ImproveB

    ImproveA (already improving): mean of available z metrics
      - ΔROE_TTM (roe_ttm - roe_ttm_1y)
      - ΔOPM_TTM (opmargin_ttm - opmargin_ttm_1y)
      - FCF positive shift (if available): sign + magnitude

    ImproveB (sustainability): mean of available z metrics
      - CashEarningsQuality = CFO_TTM / NetProfit_TTM (higher better)
      - Accrual = (NetProfit_TTM - CFO_TTM)/TotalAssets (lower better -> negative z)

    Notes:
    - All metrics winsorize + z-score within industry (IND) for stability.
    """
    df = df0.copy()

    # ImproveA components
    comps_A = []
    if {"ROE_TTM", "ROE_TTM_1Y"}.issubset(df.columns):
        df["D_ROE"] = df["ROE_TTM"] - df["ROE_TTM_1Y"]
        comps_A.append("D_ROE")
    if {"OPM_TTM", "OPM_TTM_1Y"}.issubset(df.columns):
        df["D_OPM"] = df["OPM_TTM"] - df["OPM_TTM_1Y"]
        comps_A.append("D_OPM")
    if {"FCF_TTM", "FCF_TTM_1Y"}.issubset(df.columns):
        # scale by mkt cap so cross-sectional comparable
        df["FCF_SHIFT"] = (df["FCF_TTM"] - df["FCF_TTM_1Y"]) / df["MKT_CAP"].replace(0, np.nan)
        comps_A.append("FCF_SHIFT")

    # ImproveB components
    comps_B = []
    if {"NCF_OPER_TTM", "NETPROFIT_TTM"}.issubset(df.columns):
        df["CASH_Q"] = df["NCF_OPER_TTM"] / df["NETPROFIT_TTM"].replace(0, np.nan)
        comps_B.append("CASH_Q")
    if {"NCF_OPER_TTM", "NETPROFIT_TTM", "TOT_ASSETS"}.issubset(df.columns):
        df["ACCRUAL"] = (df["NETPROFIT_TTM"] - df["NCF_OPER_TTM"]) / df["TOT_ASSETS"].replace(0, np.nan)
        comps_B.append("ACCRUAL")

    def _industry_z(col: str, higher_better: bool = True) -> pd.Series:
        tmp = df[[col, "IND"]].copy()
        tmp[col] = tmp[col].astype(float)
        tmp[col] = group_winsorize_z(tmp, col, "IND", q=q)
        return tmp[col] if higher_better else -tmp[col]

    if len(comps_A) == 0:
        improveA = pd.Series(index=df.index, data=0.0)
    else:
        A_zs = []
        for c in comps_A:
            A_zs.append(_industry_z(c, higher_better=True))
        improveA = pd.concat(A_zs, axis=1).mean(axis=1)

    if len(comps_B) == 0:
        improveB = pd.Series(index=df.index, data=0.0)
    else:
        B_zs = []
        for c in comps_B:
            if c == "ACCRUAL":
                B_zs.append(_industry_z(c, higher_better=False))  # lower accrual better
            else:
                B_zs.append(_industry_z(c, higher_better=True))
        improveB = pd.concat(B_zs, axis=1).mean(axis=1)

    improve = 0.6 * improveA + 0.4 * improveB
    # Finally cross-sectional z (so top-level comparable)
    improve = winsorize_series(improve, q=q)
    return zscore(improve)


# IC/IR weighting

In [ ]:
def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    df = pd.concat([x, y], axis=1).dropna()
    if df.shape[0] < 30:
        return np.nan
    ic, _ = spearmanr(df.iloc[:, 0], df.iloc[:, 1])
    return float(ic)


def calc_ir_weights(ic_hist: Dict[str, List[float]], window: int, floor: float) -> Dict[str, float]:
    wts = {}
    for k, lst in ic_hist.items():
        arr = pd.Series(lst).dropna()
        arr = arr.iloc[-window:] if len(arr) > window else arr
        if len(arr) < max(6, window // 2):
            wts[k] = np.nan
            continue
        mu = arr.mean()
        sd = arr.std(ddof=0)
        ir = mu / sd if sd > 0 else np.nan
        wts[k] = ir

    # convert IR -> weights (truncate at floor)
    irs = pd.Series(wts).astype(float)
    irs = irs.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    irs = irs.clip(lower=floor)

    if irs.sum() <= 0:
        # fallback equal weight
        out = {k: 1.0 / len(irs) for k in irs.index}
        return out

    out = (irs / irs.sum()).to_dict()
    return out


# Main pipeline

In [ ]:
def run_backtest(cfg: Config):
    wind_start()

    reb_dates = month_end_trading_days(cfg.start_date, cfg.end_date)
    print(f"Rebalance months: {len(reb_dates)} from {reb_dates[0]} to {reb_dates[-1]}")

    # IC history
    ic_hist = {"VALUE": [], "LOWPRICE": [], "LOWATTN": [], "IMPROVE": []}

    # backtest outputs
    positions = {}  # date -> list of stocks
    factor_weights = {}  # date -> dict
    monthly_rets = []
    turnovers = []

    prev_hold = None

    for i, d in enumerate(reb_dates[:-1]):
        d_next = reb_dates[i + 1]

        # more conservative financial date to avoid lookahead
        fin_date = wind_tdaysoffset(-cfg.fin_lag_trading_days, d)

        # 1) Universe
        u = get_universe_csi1800(d, cfg.index_codes)
        if not u:
            print(f"[{d}] No universe found. Skip.")
            continue
        # 2) Basic snapshot fields
        # Industry SW1: try commonly used names
        ind_field = None
        for cand in ["industry_sw", "industry_sw1", "industry_swcode", "industry_sw_level1"]:
            try:
                tmp = wss_df(u[:20], [cand], f"tradeDate={d}")
                if not tmp.empty and tmp[cand].notna().sum() > 0:
                    ind_field = cand
                    break
            except Exception:
                continue
        if ind_field is None:
            # final fallback: Wind industry_citic (if SW not available)
            ind_field = "industry_citic"

        # Market + valuation fields (most stable)
        fields_base = [
            "close", "mkt_cap_ard", ind_field,
            "pe_ttm", "pb_lf", "dividendyield2",
            "turn"
        ]

        base = wss_df(u, fields_base, f"tradeDate={d};PriceAdj=F;Fill=Previous")

        base = base.rename(columns={
            "close": "PRICE",
            "mkt_cap_ard": "MKT_CAP",
            ind_field: "IND",
            "pe_ttm": "PE_TTM",
            "pb_lf": "PB_LF",
            "dividendyield2": "DIVIDENDYIELD2",
            "turn": "TURN",
        })

        # 20D amount for attention fallback (compute by daily amt average)
        # We'll fetch last 20 trading days ending at d
        d_20 = wind_tdaysoffset(-20, d)
        try:
            amt_daily = wsd_df(list(base.index), "amt", d_20, d, "PriceAdj=F;Fill=Previous")
            if not amt_daily.empty:
                 # reindex ensure align
                base["AMT_20D"] = amt_daily.mean(axis=0).reindex(base.index)
            else:
                base["AMT_20D"] = np.nan
        except Exception:
            base["AMT_20D"] = np.nan

        # financial classification
        base["IS_FIN"] = classify_financial_by_sw1(base["IND"])

        # 3) Financial fields (tradeDate uses fin_date for conservatism)
        # Candidates for EV/EBIT/CFO/FCF etc.
        # If some are missing, Wind returns NaN; code will still run.
        fields_fin = [
            "ev", "ebit_ttm",
            "net_cash_flows_oper_act_ttm",
            "net_profit_is_ttm",
            "tot_assets",
            "roe_ttm",
            "opmargin_ttm",
            "free_cash_flow_ttm"
        ]
        fin = wss_df(list(base.index), fields_fin, f"tradeDate={fin_date};Fill=Previous")

        fin = fin.rename(columns={
            "ev": "EV",
            "ebit_ttm": "EBIT_TTM",
            "net_cash_flows_oper_act_ttm": "NCF_OPER_TTM",
            "net_profit_is_ttm": "NETPROFIT_TTM",
            "tot_assets": "TOT_ASSETS",
            "roe_ttm": "ROE_TTM",
            "opmargin_ttm": "OPM_TTM",
            "free_cash_flow_ttm": "FCF_TTM",
        })

        # 1Y ago financial fields for delta
        fin_1y_date = wind_tdaysoffset(-252, fin_date)
        fin_1y = wss_df(list(base.index), ["roe_ttm", "opmargin_ttm", "free_cash_flow_ttm"],
                        f"tradeDate={fin_1y_date};Fill=Previous")
        fin_1y = fin_1y.rename(columns={
            "roe_ttm": "ROE_TTM_1Y",
            "opmargin_ttm": "OPM_TTM_1Y",
            "free_cash_flow_ttm": "FCF_TTM_1Y",
        })

        df = base.join(fin, how="left").join(fin_1y, how="left")

        # 4) Low attention: try to auto-detect field names
        coverage_field = pick_first_valid_field(
            list(df.index),
            candidates=["est_analystnum", "analyst_num", "analyst_coverage", "west_analystnum"],
            options=f"tradeDate={d};Fill=Previous"
        )
        report_field = pick_first_valid_field(
            list(df.index),
            candidates=["research_report_num", "report_num", "research_report_count", "west_reportnum"],
            options=f"tradeDate={d};Fill=Previous"
        )

        if coverage_field:
            cov = wss_df(list(df.index), [coverage_field], f"tradeDate={d};Fill=Previous")
            df[coverage_field] = cov[coverage_field]
        if report_field:
            rep = wss_df(list(df.index), [report_field], f"tradeDate={d};Fill=Previous")
            df[report_field] = rep[report_field]
        
        start_hist = wind_tdaysoffset(-22 * cfg.ts_window_months, d)
        
        # 1. Price History (需复权 PriceAdj=F，反映真实涨跌)
        hist_price = wsd_df(list(df.index), "close", start_hist, d, "Period=M;PriceAdj=F")
        # 2. PE History
        hist_pe = wsd_df(list(df.index), "pe_ttm", start_hist, d, "Period=M;Fill=Previous")
        # 3. PB History
        hist_pb = wsd_df(list(df.index), "pb_lf", start_hist, d, "Period=M;Fill=Previous")

        # 5) Construct factors
        # VALUE
        value = compute_value_factor_optimized(df, hist_pe, hist_pb, cfg)

        # LOWPRICE (neutralized residual)
        lowprice = compute_lowprice_factor_optimized(df, hist_price, cfg)

        # LOWATTN
        lowattn = compute_attention_factor(
            df, q=cfg.winsor_q, min_ind_n=cfg.min_industry_n,
            coverage_col=coverage_field, report_col=report_field
        )

        # IMPROVE
        improve = compute_improve_factor(df, q=cfg.winsor_q)

        # Ensure cross-sectional z for top-level comparability
        fac = pd.DataFrame({
            "VALUE": zscore(winsorize_series(value, q=cfg.winsor_q)),
            "LOWPRICE": zscore(winsorize_series(lowprice, q=cfg.winsor_q)),
            "LOWATTN": zscore(winsorize_series(lowattn, q=cfg.winsor_q)),
            "IMPROVE": zscore(winsorize_series(improve, q=cfg.winsor_q)),
        }, index=df.index)

        # 6) Compute next-month returns for IC and backtest
        # Use close price at d and d_next (no adjustment)
        wts = calc_ir_weights(ic_hist, window=cfg.ic_window, floor=cfg.ir_floor)
        factor_weights[d] = wts
        total = pd.Series(0.0, index=fac.index)
        for k, wt in wts.items():
            if k in fac.columns:
                total = total + wt * fac[k]
        
        picks = total.sort_values(ascending=False).head(cfg.n_select).index.tolist()
        positions[d] = picks

        px0 = wss_df(list(fac.index), ["close"], f"tradeDate={d};PriceAdj=F;Fill=Previous").rename(columns={"close": "P0"})["P0"]
        px1 = wss_df(list(fac.index), ["close"], f"tradeDate={d_next};PriceAdj=F;Fill=Previous").rename(columns={"close": "P1"})["P1"]
        fwd_ret = (px1 / px0 - 1.0).replace([np.inf, -np.inf], np.nan)

        # 7) Update IC history
        current_ics = {}
        for k in ic_hist.keys():
            ic_val = spearman_ic(fac[k], fwd_ret)
            ic_hist[k].append(ic_val)
            current_ics[k] = ic_val

        # 8) Monthly portfolio return (equal weight)
        if prev_hold is None:
            turnover = 1.0
        else:
            prev = set(prev_hold)
            curr = set(picks)
            overlap = len(prev & curr)
            turnover = 1.0 - overlap / cfg.n_select
        turnovers.append(turnover)

        r = fwd_ret.reindex(picks).mean(skipna=True)
        r_net = r - cfg.cost_per_turnover * turnover
        monthly_rets.append((d_next, r_net))

        prev_hold = picks

        # 打印日志：权重使用历史数据，IC展示当前预测能力
        print(f"[{d}] N={len(df)} select={len(picks)} "
              f"ret(next)={r_net:.4f} turnover={turnover:.2%} wts={wts} next_ICs={current_ics}")

    # Results
    ret_s = pd.Series({k: v for k, v in monthly_rets}).sort_index()
    ret_s.index = pd.to_datetime(ret_s.index)

    # Perf stats (monthly)
    ann_ret = (1 + ret_s).prod() ** (12 / len(ret_s)) - 1
    ann_vol = ret_s.std(ddof=0) * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan

    cum = (1 + ret_s).cumprod()
    dd = cum / cum.cummax() - 1
    mdd = dd.min()

    out = {
        "monthly_returns": ret_s,
        "cum": cum,
        "max_drawdown": mdd,
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "positions": positions,
        "factor_weights": factor_weights,
        "turnover": pd.Series(turnovers, index=pd.to_datetime(reb_dates[1:len(turnovers)+1]))
    }
    return out

# Run

In [ ]:
if __name__ == "__main__":
    res = run_backtest(CFG)

    print("\n==== Performance (monthly backtest) ====")
    print(f"Ann.Return : {res['ann_return']:.2%}")
    print(f"Ann.Vol    : {res['ann_vol']:.2%}")
    print(f"Sharpe     : {res['sharpe']:.3f}")
    print(f"MaxDD      : {res['max_drawdown']:.2%}")

    # Save outputs
    res["monthly_returns"].to_csv("monthly_returns.csv", encoding="utf-8-sig")
    res["turnover"].to_csv("monthly_turnover.csv", encoding="utf-8-sig")

    # Example: export last rebalance picks
    if res["positions"]:
        last_date = sorted(res["positions"].keys())[-1]
        pd.Series(res["positions"][last_date], name="wind_code").to_csv(
            f"picks_{last_date}.csv", index=False, encoding="utf-8-sig"
        )
        print(f"\nSaved: monthly_returns.csv, monthly_turnover.csv, picks_{last_date}.csv")
    else:
        print("No positions generated.")